# Volume 1, Chapter 2: Primes (Python companion)

Exercises first, solutions at the end. Computer-science threads: the **sieve of Eratosthenes** (and how to make it fast
with NumPy), and the **Lucas–Lehmer test**, which finds the largest known primes.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from mpmath import li
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10102)
seed_sympy(20102)


## Before the experiment: A sieve is an argument with a picture

Guess which is closer to $\pi(10^7)$: $x/\log x$ or $\operatorname{Li}(x)$. A graph can suggest an asymptotic law, but even ten million cases cannot prove a limit. For Bertrand, remember the special endpoint $n=1$.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P2.1 (★) The sieve.** Implement the sieve of Eratosthenes with a NumPy boolean array, list the primes below $10^7$, and time it. Why is it enough to cross out multiples of $p$ starting at $p^2$?

In [ ]:
# Your code here


**P2.2 (★) Counting primes.** Plot $\pi(x)$, $x/\ln x$ and $\operatorname{Li}(x) = \int_2^x \frac{dt}{\ln t}$ for $x \le 10^7$, and print the ratios $\pi(x)\ln x / x$ and $\pi(x)/\operatorname{Li}(x)$ at powers of 10. Which approximation is better?

In [ ]:
# Your code here


**P2.3 (★★) Euler and Mertens.** Compare $\sum_{p \le x} 1/p$ with $\ln\ln x$. The difference tends to the *Meissel–Mertens constant* $M \approx 0.2615$; estimate it.

In [ ]:
# Your code here


**P2.4 (★★) Bertrand, checked.** Verify Bertrand's postulate for all $n \le 10^6$ using the sieve, and verify numerically the inequality $\frac n3\ln 4 > (1 + \sqrt{2n})\ln(2n)$ for $468 \le n \le 10^7$ used in Erdős' proof. Also check $\prod_{p \le x} p < 4^x$ (in logarithms: $\theta(x) = \sum_{p\le x}\ln p < x\ln 4$).

In [ ]:
# Your code here


**P2.5 (★★) Fermat numbers.** Fermat believed every $\mathcal F_n = 2^{2^n}+1$ is prime. Find the smallest prime factor of $\mathcal F_5$ by trial division over candidates of the form $k\cdot 2^{7}+1$ (for $n\ge2$, a prime divisor of $\mathcal F_n$ is $1\pmod{2^{n+2}}$: its order gives the first factor of $2$, and the supplementary law for $2$ in Chapter 6 gives the extra one).

In [ ]:
# Your code here


**P2.6 (★★★) CS application: Lucas–Lehmer.** For an odd prime $p$, $M_p = 2^p - 1$ is prime if and only if $s_{p-2} \equiv 0 \pmod{M_p}$, where $s_0 = 4$ and $s_{k+1} = s_k^2 - 2$. Implement it and find all Mersenne primes $M_p$ with $p < 3000$. (The proof uses the arithmetic of $\mathbb Z[\sqrt 3]$; this is a practical test for Mersenne primes, not a test for arbitrary integers.)

In [ ]:
# Your code here


---
## Solutions

### P2.1

In [ ]:
def sieve(n):
    is_p = np.ones(n + 1, dtype=bool)
    is_p[:2] = False
    for p in range(2, math.isqrt(n) + 1):
        if is_p[p]:
            is_p[p * p :: p] = False
    return np.flatnonzero(is_p)

import time
t = time.perf_counter()
P = sieve(10**7)
print(f"{len(P)} primes below 10^7 in {time.perf_counter() - t:.2f} s; last: {P[-1]}")

A composite $m$ has a prime factor $\le\sqrt m$, so a multiple $kp$ with $k < p$ was already crossed out when its smaller prime factor was processed.

### P2.2

In [ ]:
xs = np.unique(np.logspace(1, 7, 300).astype(int))
pi = np.searchsorted(P, xs, side="right")
Li = np.array([float(li(x) - li(2)) for x in xs])
plt.loglog(xs, pi, label=r"$\pi(x)$"); plt.loglog(xs, xs / np.log(xs), label=r"$x/\ln x$"); plt.loglog(xs, Li, "--", label="Li(x)")
plt.legend(); plt.show()
for k in range(2, 8):
    x = 10**k
    n = np.searchsorted(P, x, side="right")
    print(f"x = 10^{k}: pi = {n:7d}   pi ln x / x = {n * math.log(x) / x:.4f}   pi / Li = {n / float(li(x) - li(2)):.6f}")

$\operatorname{Li}(x)$ is far better: at $10^7$ it is off by about $0.05\%$, while $x/\ln x$ is off by $7\%$. The prime number theorem (Volume 5) says both ratios tend to 1.

### P2.3

In [ ]:
recip = np.cumsum(1.0 / P)
for k in range(2, 8):
    x = 10**k
    n = np.searchsorted(P, x, side="right")
    print(f"x = 10^{k}: sum 1/p = {recip[n - 1]:.5f}   ln ln x = {math.log(math.log(x)):.5f}   difference = {recip[n - 1] - math.log(math.log(x)):.5f}")

### P2.4

In [ ]:
is_p = np.zeros(2 * 10**6 + 1, dtype=bool)
is_p[P[P <= 2 * 10**6]] = True
count = np.cumsum(is_p)
n = np.arange(1, 10**6 + 1)
assert np.all(count[2 * n] - count[n] > 0)
print("The interval (n, 2n] contains a prime for all 1 <= n <= 10^6 (n = 1 is the endpoint exception)")
assert np.all(count[2*n[1:] - 1] - count[n[1:]] > 0)

m = np.arange(468, 10**7, dtype=float)
assert np.all(m / 3 * math.log(4) > (1 + np.sqrt(2 * m)) * np.log(2 * m))
m = np.arange(5, 468, dtype=float)
print("the inequality fails for n =", int(m[~(m / 3 * math.log(4) > (1 + np.sqrt(2 * m)) * np.log(2 * m))].max()), "and holds from 468 on")

theta = np.cumsum(np.log(P))
assert np.all(theta < P * math.log(4))
print("theta(x) < x ln 4 at every prime below 10^7; theta(10^7)/10^7 =", theta[-1] / 10**7)

### P2.5

In [ ]:
F5 = 2**32 + 1
k = 1
while F5 % (k * 2**7 + 1):
    k += 1
d = k * 2**7 + 1
print(f"F_5 = {F5} = {d} x {F5 // d}")

### P2.6

In [ ]:
def lucas_lehmer(p):
    M = (1 << p) - 1
    s = 4
    for _ in range(p - 2):
        s = (s * s - 2) % M
    return s == 0

odd_primes = [int(p) for p in P if 2 < p < 3000]
mersenne = [2] + [p for p in odd_primes if lucas_lehmer(p)]
print("Mersenne prime exponents below 3000:", mersenne)

## Zoom in: relative errors

The three counting curves nearly overlap at large $x$. Subtracting the exact count and dividing by it makes their differences visible. A logarithmic horizontal scale gives small and large ranges room to speak.


In [ ]:
approx_log = xs / np.log(xs)
plt.figure(figsize=(7, 4))
plt.semilogx(xs, (approx_log - pi) / pi, "-", label=r"$(x/\log x-\pi(x))/\pi(x)$")
plt.semilogx(xs, (Li - pi) / pi, "--", label=r"$(\operatorname{Li}(x)-\pi(x))/\pi(x)$")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("x"); plt.ylabel("relative error"); plt.title("A close approximation deserves a closer look")
plt.legend(); plt.tight_layout(); plt.show()


## One more question before closing

Find the largest observed error of each prime-count approximation on this grid. Does a relative-error plot tell a different story from a count plot?

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
